In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
import numpy as np
from joblib import dump
import sys
import os

In [2]:
sys.path.append(os.path.abspath(".."))

from Data_Preparation.soil_temp_data_preparation import (
    create_soil_temp_pipeline,
    prepare_soil_temp_data
)

In [3]:
def evaluate_xgb(X_train, y_train, X_dev, y_dev):
    print("Evaluating XGBoost Regressor...")

    param_grid = {
        'algo__n_estimators': [100, 300],
        'algo__max_depth': [3, 5],
        'algo__learning_rate': [0.05, 0.1],
        'algo__subsample': [0.8, 1.0],
    }

    pipeline = create_soil_temp_pipeline()

    pipeline_with_algo = Pipeline(steps=[
        ('preprocessor', pipeline),
        ('algo', XGBRegressor(objective='reg:squarederror', random_state=42))
    ])

    grid_search = GridSearchCV(
        pipeline_with_algo,
        param_grid,
        cv=3,
        scoring='neg_mean_squared_error',
        verbose=1,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    best_estimator = grid_search.best_estimator_

    try:
        model = best_estimator.named_steps["algo"]
        feature_names = X_train.columns
        importances = model.feature_importances_

        feature_df = pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        }).sort_values(by="Importance", ascending=False)

        print("\nTop 10 Most Important Features:")
        print(feature_df.head(10))

    except Exception as e:
        print("Could not extract feature importances:", e)

    y_pred = best_estimator.predict(X_dev)
    rmse = np.sqrt(mean_squared_error(y_dev, y_pred))
    mae = mean_absolute_error(y_dev, y_pred)
    r2 = r2_score(y_dev, y_pred)

    print("Grid searching is done!")
    print("Best score (neg MSE):", grid_search.best_score_)
    print("Best hyperparameters:")
    print(grid_search.best_params_)

    return best_estimator, rmse, mae, r2

In [4]:
def evaluate_metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mean_target = np.mean(y_true)

    print(f"\n📊 {label} Set Performance:")
    print(f"Mean of y_{label.lower()}: {mean_target:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")

    return rmse, mae, r2

In [5]:
print("\n🚀 Evaluating model for: Soil Temperature")

X_train, X_dev, X_test, y_train, y_dev, y_test = prepare_soil_temp_data()


🚀 Evaluating model for: Soil Temperature


In [6]:
best_model, _, _, _ = evaluate_xgb(X_train, y_train, X_dev, y_dev)

Evaluating XGBoost Regressor...
Fitting 3 folds for each of 16 candidates, totalling 48 fits

Top 10 Most Important Features:
                  Feature  Importance
7         air_temperature    0.926746
4       total_evaporation    0.021215
9                   month    0.014795
14     district_northeast    0.009742
5   potential_evaporation    0.004857
1               longitude    0.003307
3                  runoff    0.003133
8                    year    0.003036
10                    day    0.002856
6    dewpoint_temperature    0.002433
Grid searching is done!
Best score (neg MSE): -0.023001164597642756
Best hyperparameters:
{'algo__learning_rate': 0.1, 'algo__max_depth': 5, 'algo__n_estimators': 300, 'algo__subsample': 0.8}


In [7]:
print("✅ Data Split Shapes:")
print("  X_train:", X_train.shape)
print("  X_dev:", X_dev.shape)
print("  X_test:", X_test.shape)
print("  y_train:", y_train.shape)
print("  y_dev:", y_dev.shape)
print("  y_test:", y_test.shape)

✅ Data Split Shapes:
  X_train: (1679864, 20)
  X_dev: (419967, 20)
  X_test: (10000, 20)
  y_train: (1679864,)
  y_dev: (419967,)
  y_test: (10000,)


In [8]:
y_train_pred = best_model.predict(X_train)
y_dev_pred = best_model.predict(X_dev)
y_test_pred = best_model.predict(X_test)

In [9]:
evaluate_metrics(y_train, y_train_pred, "Train")
evaluate_metrics(y_dev, y_dev_pred, "Dev")
evaluate_metrics(y_test, y_test_pred, "Test")


📊 Train Set Performance:
Mean of y_train: -0.0002
RMSE: 0.1509
MAE: 0.1131
R²: 0.9772

📊 Dev Set Performance:
Mean of y_dev: 0.0007
RMSE: 0.1513
MAE: 0.1135
R²: 0.9771

📊 Test Set Performance:
Mean of y_test: 0.0005
RMSE: 0.1511
MAE: 0.1135
R²: 0.9778


(0.15108431876735381, 0.11346185323107386, 0.977815274707157)

In [10]:
dump(best_model, "../models/soil_temperature_model.joblib")

print("✅ Model saved as: ../models/soil_temperature_model.joblib")

✅ Model saved as: ../models/soil_temperature_model.joblib
